# Smart Delivery ETA & Delay Prediction Engine
### Exploratory Data Analysis, Feature Engineering & Prototyping Notebook

This notebook provides exploratory insights, distribution checks, and prototyping workflows for:
1. **Delivery ETA (Regression)**
2. **Delay Probability & Risk Scoring (Binary Classification)**

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import sqlite3

# Ensure project src is in sys.path
sys.path.append(os.path.abspath("../src"))
from db_manager import DeliveryDatabase

db = DeliveryDatabase("../database/delivery_orders.db")
kpis = db.get_kpis()
print("KPI Summary:", kpis)

## 1. Distribution Analysis: Distance & Delivery Duration

In [ ]:
df_dist = db.get_distance_vs_delivery_time()
print(df_dist)

## 2. Weather and Traffic Delay Dynamics

In [ ]:
df_traffic = db.get_traffic_impact()
df_weather = db.get_weather_impact()
print("Traffic Impact:\n", df_traffic)
print("\nWeather Impact:\n", df_weather)

## 3. Model Pipeline Prototyping & Inference Test

In [ ]:
import joblib
from models_def import ETAPipeline, DelayPipeline
from explainability import DeliveryExplainer

eta_pipe = joblib.load("../models/eta_pipeline.joblib")
delay_pipe = joblib.load("../models/delay_pipeline.joblib")
explainer = DeliveryExplainer("../models/eta_pipeline.joblib", "../models/delay_pipeline.joblib")

sample_order = pd.DataFrame([{
    'restaurant_id': 'REST_005',
    'customer_id': 'CUST_0010',
    'delivery_partner_id': 'DP_020',
    'order_time': '2026-09-20 20:00:00',
    'day_of_week': 'Friday',
    'distance_km': 8.5,
    'restaurant_prep_time': 28.0,
    'order_size': 'Medium',
    'item_count': 5,
    'order_value': 62.0,
    'traffic_level': 'High',
    'weather': 'Rainy',
    'temperature': 22.0,
    'precipitation': 12.0,
    'active_delivery_partners': 45,
    'orders_last_30_min': 95,
    'peak_hour': 1,
    'delivery_partner_experience': 'Intermediate',
    'delivery_partner_experience_months': 8
}])

eta_res = eta_pipe.predict_with_interval(sample_order)
delay_res = delay_pipe.predict_risk(sample_order)
exp_res = explainer.explain_order(sample_order)

print(f"Predicted ETA: {eta_res['predicted_eta']} min ({eta_res['range_str']})")
print(f"Delay Probability: {delay_res['probability_pct']}% | Risk Level: {delay_res['risk_level']}")
print("Top Drivers:", [f['feature'] for f in exp_res['top_factors'][:3]])